In [53]:
from langgraph.graph import StateGraph,START, END
from typing import  TypedDict

In [54]:
# Defining the state 
class BMIState(TypedDict): # we are inheriting from TypedDict to create a structured state representation
    weight: float  # in kilograms
    height: float  # in meters
    bmi : float     # Body Mass Index




In [55]:
# functions :
def calculate_bmi(state : BMIState) -> BMIState: #here we are defining a function that takes in a BMIState and returns a new BMIState with the bmi calculated
    weight = state['weight']
    height  = state['height']
    bmi = weight / (height ** 2) # BMI formula
    state['bmi'] = round(bmi, 2)
    return state;


def label_bmi(state : BMIState) -> BMIState :
    bmi = state['bmi']
    if bmi < 18.5:
        category = 'Underweight'
    elif 18.5 <= bmi < 25:
        category = 'Normal weight'
    elif 25 <= bmi < 30:
        category = 'Overweight'
    else:
        category = 'Obese'
    state['category'] = category
    return state;


In [56]:
# defining the graph
graph = StateGraph(BMIState) # we are creating a state graph that will use the BMIState as its state representation 

# add nodes to the graph
graph.add_node('calculate_bmi' , calculate_bmi) # we are adding a node with name and the function that it will execute when this node is reached in the graph

# add edges to the graph
# as this is a simple graph with only one node, we don't need to add any edges. In a more complex graph, we would add edges to define the flow between nodes.
graph.add_edge(START, 'calculate_bmi') # we are adding an edge from a 'start' node to the 'calculate_bmi' node to indicate that the graph should start by executing the calculate_bmi function
graph.add_edge('calculate_bmi', END) # we are adding an edge from the 'calculate_bmi' node to an 'end' node to indicate that after executing the calculate_bmi function, the graph should end


# compile the graph
workflow = graph.compile() # this will prepare the graph for execution by checking for any errors and setting up the necessary data structures

In [57]:
# execute the graph
initial_state =  {'weight': 70, 'height': 1.75}
output = workflow.invoke(initial_state) # we are invoking the workflow with an initial state that includes weight and height. The workflow will execute the calculate_bmi function and return the final state with the calculated bmi.
print(output['bmi'])

22.86


In [58]:

class BMIState2(TypedDict):
    weight: float
    height: float
    bmi: float
    category: str

In [59]:

graph = StateGraph(BMIState2)

graph.add_node('calculate_bmi' , calculate_bmi)
graph.add_node('categorize_bmi' , label_bmi)

graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'categorize_bmi')
graph.add_edge('categorize_bmi', END)

workflow = graph.compile()
initial_state =  {'weight': 70, 'height': 1.75}
output = workflow.invoke(initial_state)
print(output['bmi'])
print(output['category'])


22.86
Normal weight


### basic LLM workflow 

In [79]:
# a simple workflow where we will ask question to llm and it will ans  and then we will end .
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os


In [80]:
load_dotenv() # this will load the environment variables from the .env file, including our API key for the language model
model = ChatGoogleGenerativeAI(api_key=os.getenv("GEMINI_API_KEY") , model="gemini-3-flash-preview") # we are creating an instance of the ChatGoogleGenerativeAI model using the API key from the environment variables

In [73]:
# create a state 
class LLMState( TypedDict) : 
    question : str 
    ans : str 


In [92]:
def LLMqa(state : LLMState) -> LLMState :
    question = state['question']

    # from a prompt 
    prompt = f"Answer the following question : {question}"

    # calling the llm 
    answer = model.invoke(prompt).content[0]['text']

    # we will store the answer in the state and return it
    state['ans'] = answer
    return state

In [93]:
graph = StateGraph(LLMState)

graph.add_node('ask_question' , LLMqa)

graph.add_edge(START, 'ask_question')
graph.add_edge('ask_question', END)

workflow = graph.compile()


In [95]:
initial_state =  {'question': "What is the capital of chaina?"}
result = workflow.invoke(initial_state)
print(result['ans'])

The capital of **China** is **Beijing**.


### Prompt Chaining : 

In [97]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os

In [98]:
load_dotenv()

True

In [99]:
# defining the state 
class BlogState(TypedDict):
    title : str 
    outline : str 
    content : str 


In [ ]:
def create_outline(state : BlogState) -> BlogState :
    title = state['title']
    prompt = f'Generate an outline for a blog on the topic: {title}'
    outline = model.invoke(prompt).content[0]['text']
    state['outline'] = outline
    return state;


def create_blog(state : BlogState) -> BlogState :
    title = state["title"]
    outline = state['outline']
    prompt = f'write a detailed blog on the title {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content[0]['text']
    state['content'] = content
    return state




In [104]:
graph = StateGraph(BlogState)

graph.add_node('create_outline' , create_outline )
graph.add_node('create_blog' , create_blog )

graph.add_edge(START , 'create_outline' )
graph.add_edge('create_outline', 'create_blog' )
graph.add_edge('create_blog', END )

workflow = graph.compile()

In [105]:
initial_state = {"title" : 'Rise of Ai in india'}
final_state =workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of Ai in india', 'outline': 'This outline provides a comprehensive structure for a blog post titled **"The Rise of AI in India: From IT Hub to Global AI Superpower."**\n\n---\n\n### **Blog Title Ideas**\n*   *The Saffron Silicon Valley: How India is Leading the AI Revolution*\n*   *AI for All: Decoding India’s Artificial Intelligence Transformation*\n*   *From Back-Office to Brain-Trust: The Rise of AI in India*\n\n---\n\n### **I. Introduction**\n*   **The Hook:** Start with a striking statistic (e.g., India’s projected AI market growth or the number of AI startups).\n*   **The Narrative Shift:** Move from India’s reputation as the "world’s back office" (outsourcing) to its new identity as a global hub for AI innovation.\n*   **Thesis Statement:** India is uniquely positioned to lead the AI race due to its vast data pool, government support, and massive talent base.\n\n### **II. The Catalyst: Why India? Why Now?**\n*   **Digital Public Infrastructure (DPI):** How the "I

In [107]:
print(final_state['outline'])

This outline provides a comprehensive structure for a blog post titled **"The Rise of AI in India: From IT Hub to Global AI Superpower."**

---

### **Blog Title Ideas**
*   *The Saffron Silicon Valley: How India is Leading the AI Revolution*
*   *AI for All: Decoding India’s Artificial Intelligence Transformation*
*   *From Back-Office to Brain-Trust: The Rise of AI in India*

---

### **I. Introduction**
*   **The Hook:** Start with a striking statistic (e.g., India’s projected AI market growth or the number of AI startups).
*   **The Narrative Shift:** Move from India’s reputation as the "world’s back office" (outsourcing) to its new identity as a global hub for AI innovation.
*   **Thesis Statement:** India is uniquely positioned to lead the AI race due to its vast data pool, government support, and massive talent base.

### **II. The Catalyst: Why India? Why Now?**
*   **Digital Public Infrastructure (DPI):** How the "India Stack" (Aadhaar, UPI, DigiLocker) created the digital rai